# INF8225 — Projet : Whisper pour le français québécois
**École Polytechnique de Montréal — Équipe 12**  
Hamza El Haddad · Jules Gouy

---

## 🔄 Stratégie de reprise multi-sessions

Chaque étape terminée est sauvegardée comme **WandB Artifact** (résultats JSON + weights + figures).  
À chaque nouvelle session Colab, exécutez **§1 puis §0** — tout ce qui a déjà été calculé
sera récupéré automatiquement depuis WandB sans recalcul.

| Artifact WandB | Contenu sauvegardé |
|---|---|
| `baseline` | WER/CER par variante et dataset (JSON) |
| `finetuning` | Weights du modèle + métriques test + courbes |
| `ablation-lr` | WER val par taux d'apprentissage (JSON) |
| `ablation-freeze` | WER val gel vs libre (JSON) |
| `ablation-augment` | WER val avec augmentation (JSON) |
| `wav2vec` | WER/CER wav2vec 2.0 (JSON) |
| `figures` | Toutes les figures PNG |

## Table des matières
**[§1 Setup](#section1)** → **[§0 Reprise](#section0)** → §2 Datasets → §3 EDA → §4 Baseline → §5 Erreurs → §6 Fine-tuning → §7 Ablations → §8 wav2vec → §9 Synthèse

---
## 1. Setup & dépendances <a id='section1'></a>
> ▶️ **Toujours exécuter cette section en premier**, même lors d'une reprise.

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "⚠️  Pas de GPU — Runtime > Change runtime type > T4 GPU")

In [ ]:
%%capture
!pip install -q \
    transformers==4.40.0 datasets==2.19.0 evaluate==0.4.2 jiwer==3.0.4 \
    accelerate==0.29.3 librosa==0.10.1 soundfile==0.12.1 wandb==0.17.0 \
    audiomentations==0.36.2 sentencepiece bitsandbytes
print("OK")

In [ ]:
import os, re, json, random, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
from pathlib import Path
from IPython.display import Audio, display

import torch
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    WhisperTokenizer, WhisperFeatureExtractor,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback,
    Wav2Vec2ForCTC, Wav2Vec2Processor,
)
from datasets import load_dataset, Dataset, Audio as AudioFeature
import evaluate
from jiwer import wer, cer
import wandb

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
WANDB_API_KEY  = "YOUR_WANDB_API_KEY"   # <-- remplacer
WANDB_PROJECT  = "inf8225-whisper-quebec"
WANDB_ENTITY   = None                   # votre username WandB (ou None = auto)
HF_TOKEN       = "YOUR_HF_TOKEN"        # <-- remplacer (compte HF requis pour CommonVoice)

# Répertoires — le LOCAL_CKPT_DIR sur Drive survit aux redémarrages Colab
LOCAL_CKPT_DIR = "/content/drive/MyDrive/INF8225/checkpoints"
FIG_DIR        = "/content/figures"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Paramètres dataset (CV_VERSION défini en §2 selon la source disponible)
N_TEST      = 500
N_TRAIN_FT  = 2000
N_VAL_FT    = 300
SAMPLE_RATE = 16_000
MODEL_FT    = "openai/whisper-small"

wandb.login(key=WANDB_API_KEY)
print("✅ Config OK")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🗄️  UTILITAIRES DE CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════

def _art_name(step: str) -> str:
    return f"{WANDB_PROJECT}-{step}".replace("/", "-")

def artifact_exists(step: str) -> bool:
    """Retourne True si un artifact 'latest' existe déjà sur WandB pour cette étape."""
    try:
        api    = wandb.Api()
        entity = WANDB_ENTITY or api.default_entity
        api.artifact(f"{entity}/{WANDB_PROJECT}/{_art_name(step)}:latest")
        return True
    except Exception:
        return False

def save_artifact(step: str, artifact_type: str, files: Dict[str, str],
                  metadata: Optional[Dict] = None, run=None):
    """
    Sauvegarde des fichiers/dossiers comme WandB Artifact.
    files = {"nom_logique": "/chemin/local"}
    Si run=None, un run temporaire est créé et fermé automatiquement.
    """
    owns_run = run is None
    if owns_run:
        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                         name=f"ckpt-{step}", job_type="checkpoint", reinit="finish_previous")
    art = wandb.Artifact(name=_art_name(step), type=artifact_type,
                         metadata=metadata or {})
    for logical, local in files.items():
        p = Path(local)
        if p.is_dir():   art.add_dir(str(p), name=logical)
        elif p.is_file(): art.add_file(str(p), name=logical)
        else: print(f"  ⚠️  Introuvable : {local}")
    run.log_artifact(art)
    art.wait()
    print(f"  💾 Artifact '{_art_name(step)}' sauvegardé sur WandB")
    if owns_run:
        wandb.finish()

def load_artifact(step: str, dest: str) -> str:
    """Télécharge un artifact WandB et retourne le chemin local."""
    api    = wandb.Api()
    entity = WANDB_ENTITY or api.default_entity
    art    = api.artifact(f"{entity}/{WANDB_PROJECT}/{_art_name(step)}:latest")
    path   = art.download(root=dest)
    print(f"  📥 '{_art_name(step)}' → {path}")
    return path

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_fig(name: str) -> str:
    path = f"{FIG_DIR}/{name}"
    plt.savefig(path, bbox_inches='tight')
    return path

# Aperçu du statut actuel
print("\n── Statut des étapes sur WandB ──")
STEPS = ["baseline","finetuning","ablation-lr","ablation-freeze","ablation-augment","wav2vec"]
for s in STEPS:
    icon = "✅" if artifact_exists(s) else "⬜"
    print(f"  {icon}  {s}")

---
## 0. 🔄 Reprise de session <a id='section0'></a>

**Exécutez cette cellule après §1 à chaque redémarrage.**  
Elle récupère automatiquement tous les résultats déjà calculés depuis WandB  
et affiche les sections qu'il reste à exécuter.

In [ ]:
RESUME_DIR = "/content/resumed"
os.makedirs(RESUME_DIR, exist_ok=True)

# ── État global (None = pas encore calculé) ───────────────────────────────────
baseline_results        = None
ft_result               = None
FT_MODEL_PATH           = None
ablation_lr_results     = None
ablation_freeze_results = None
ablation_aug_wer        = None
w2v_fr                  = None
w2v_frca                = None

print("═" * 55)
print("       RÉCUPÉRATION DES CHECKPOINTS WANDB")
print("═" * 55)

# ── §4 Baseline ──────────────────────────────────────────────────────────────
if artifact_exists("baseline"):
    p = load_artifact("baseline", f"{RESUME_DIR}/baseline")
    baseline_results = load_json(f"{p}/baseline_results.json")
    print(f"  → {len(baseline_results)} combinaisons (variante × dataset)")
else:
    print("  ⬜ baseline — à calculer en §4")

# ── §6 Fine-tuning ────────────────────────────────────────────────────────────
if artifact_exists("finetuning"):
    p = load_artifact("finetuning", f"{RESUME_DIR}/finetuning")
    ft_result     = load_json(f"{p}/ft_result.json")
    FT_MODEL_PATH = f"{p}/model"
    print(f"  → WER test = {ft_result['wer']*100:.2f}%  |  Modèle : {FT_MODEL_PATH}")
else:
    print("  ⬜ finetuning — à calculer en §6")

# ── §7 Ablation LR ───────────────────────────────────────────────────────────
if artifact_exists("ablation-lr"):
    p = load_artifact("ablation-lr", f"{RESUME_DIR}/ablation-lr")
    # Clés JSON = strings → reconvertir en float
    ablation_lr_results = {float(k): v
                           for k, v in load_json(f"{p}/ablation_lr.json").items()}
    print(f"  → ablation LR : {ablation_lr_results}")
else:
    print("  ⬜ ablation-lr — à calculer en §7")

# ── §7 Ablation freeze ───────────────────────────────────────────────────────
if artifact_exists("ablation-freeze"):
    p = load_artifact("ablation-freeze", f"{RESUME_DIR}/ablation-freeze")
    ablation_freeze_results = load_json(f"{p}/ablation_freeze.json")
    print(f"  → ablation freeze : {ablation_freeze_results}")
else:
    print("  ⬜ ablation-freeze — à calculer en §7")

# ── §7 Ablation augmentation ─────────────────────────────────────────────────
if artifact_exists("ablation-augment"):
    p = load_artifact("ablation-augment", f"{RESUME_DIR}/ablation-augment")
    ablation_aug_wer = load_json(f"{p}/ablation_augment.json")["wer"]
    print(f"  → ablation augmentation : WER={ablation_aug_wer:.2f}%")
else:
    print("  ⬜ ablation-augment — à calculer en §7")

# ── §8 wav2vec 2.0 ───────────────────────────────────────────────────────────
if artifact_exists("wav2vec"):
    p    = load_artifact("wav2vec", f"{RESUME_DIR}/wav2vec")
    data = load_json(f"{p}/wav2vec_results.json")
    w2v_fr   = data["fr"]
    w2v_frca = data["fr-CA"]
    print(f"  → wav2vec fr={w2v_fr['wer']*100:.2f}%  fr-CA={w2v_frca['wer']*100:.2f}%")
else:
    print("  ⬜ wav2vec — à calculer en §8")

print()
todo = [s for s in STEPS if not artifact_exists(s)]
if todo:
    print(f"📋 Sections restantes : {', '.join(todo)}")
else:
    print("🎉 Toutes les étapes sont complètes — passez directement à §9 (Synthèse)")

---
## 2. Chargement des datasets <a id='section2'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Situation datasets (avril 2026) :
#   • CommonVoice (mozilla-foundation/*) → scripts .py bloqués + données
#     déplacées vers Mozilla Data Collective (auth requise) ✗
#   • Miroirs fsicoli/* → même problème de script .py ✗
#   • facebook/multilingual_librispeech "french" → Parquet natif, 0 auth ✅
#
# Stratégie retenue :
#   • fr standard  : MLS french  (test split)        → colonne 'transcript'
#   • fr-CA        : MLS french  (train split)        → proxy québécois
#                    ⚠️  Limite : MLS est du français lu (livres audio),
#                    pas spécifiquement québécois. À mentionner dans le rapport.
#
# Alternative si vous avez dl CommonVoice manuellement depuis Mozilla :
#   → remplacez les blocs MLS par la fonction load_cv_local() ci-dessous
# ══════════════════════════════════════════════════════════════════════════════

def stream_mls(split, n, offset=0):
    """
    Charge n exemples de MLS french depuis le split donné.
    offset : sauter les premiers exemples (pour simuler des splits distincts).
    Retourne un Dataset HF avec colonne 'sentence' et audio 16kHz.
    """
    ds_stream = load_dataset(
        "facebook/multilingual_librispeech", "french",
        split=split, streaming=True,
    )
    samples = []
    for i, ex in enumerate(ds_stream):
        if i < offset:
            continue
        samples.append({
            "sentence": ex["transcript"],   # MLS utilise 'transcript'
            "audio":    ex["audio"],
        })
        if len(samples) >= n:
            break
        if (len(samples)) % 100 == 0:
            print(f"  {len(samples)}/{n} …")
    return Dataset.from_list(samples).cast_column(
        "audio", AudioFeature(sampling_rate=SAMPLE_RATE)
    )

# ── fr standard (test) ────────────────────────────────────────────────────────
print("Chargement MLS french → fr test …")
fr_test = stream_mls("test", N_TEST)
cv_fr = {"test": fr_test}

# ── fr-CA proxy : train MLS décalé pour éviter overlap avec le test ───────────
# On prend train[N_TEST : N_TEST+N_TRAIN_FT] et train[...] pour val
print("\nChargement MLS french → fr-CA test (proxy) …")
frca_test = stream_mls("test",  N_TEST,    offset=N_TEST)   # 2e moitié du test

print("\nChargement MLS french → fr-CA train …")
frca_train = stream_mls("train", N_TRAIN_FT, offset=0)

print("\nChargement MLS french → fr-CA validation …")
frca_val = stream_mls("dev", N_VAL_FT, offset=0)

cv_frca = {"test": frca_test, "train": frca_train, "validation": frca_val}

print(f"\n✅ fr   test  : {len(cv_fr['test'])} ex.   (MLS french / test)")
print(f"✅ fr-CA test  : {len(cv_frca['test'])} ex.   (MLS french / test offset)")
print(f"✅ fr-CA train : {len(cv_frca['train'])} ex.  (MLS french / train)")
print(f"✅ fr-CA val   : {len(cv_frca['validation'])} ex.   (MLS french / dev)")
print(f"Colonnes : {cv_frca['test'].column_names}")
print()
print("⚠️  NOTE RAPPORT : faute d'accès à CommonVoice fr-CA, on utilise MLS")
print("   french comme proxy. Les deux splits partagent la même distribution")
print("   acoustique (livres audio, français standard). L'écart fr vs fr-CA")
print("   sera donc sous-estimé par rapport à un vrai corpus québécois.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🔁 OPTIONNEL — À utiliser si vous avez téléchargé CommonVoice manuellement
#    depuis https://commonvoice.mozilla.org/fr/datasets
#    et uploadé les .tar.gz dans MyDrive/INF8225/data/
#
# Ne PAS exécuter cette cellule si vous utilisez MLS ci-dessus.
# ══════════════════════════════════════════════════════════════════════════════
import tarfile
import soundfile as sf
from pathlib import Path

DRIVE_DATA = "/content/drive/MyDrive/INF8225/data"

def load_cv_local(archive_path, n_test=500, n_train=2000, n_val=300):
    """Charge CommonVoice depuis une archive .tar.gz téléchargée manuellement."""
    extract_dir = archive_path.replace(".tar.gz", "")
    if not os.path.exists(extract_dir):
        print(f"  Extraction … (peut prendre plusieurs minutes)")
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(path=os.path.dirname(archive_path))
    root  = next(p for p in Path(extract_dir).rglob("validated.tsv")).parent
    clips = root / "clips"

    def read_split(tsv_file, n):
        tsv = root / tsv_file
        if not tsv.exists():
            raise FileNotFoundError(f"{tsv} introuvable")
        df = pd.read_csv(tsv, sep="\t").dropna(subset=["sentence"]).head(n)
        samples = []
        for _, row in df.iterrows():
            mp3 = clips / row["path"]
            if not mp3.exists(): continue
            arr, sr = sf.read(str(mp3), dtype="float32")
            if sr != SAMPLE_RATE:
                import librosa
                arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
            samples.append({"sentence": row["sentence"],
                            "audio": {"array": arr, "sampling_rate": SAMPLE_RATE}})
        return Dataset.from_list(samples).cast_column(
            "audio", AudioFeature(sampling_rate=SAMPLE_RATE))

    return {
        "test":       read_split("test.tsv",  n_test),
        "train":      read_split("train.tsv", n_train),
        "validation": read_split("dev.tsv",   n_val),
    }

# Décommenter pour activer :
# cv_fr   = load_cv_local(f"{DRIVE_DATA}/cv-corpus-17.0-fr.tar.gz",
#                         n_test=N_TEST, n_train=0, n_val=0)
# cv_frca = load_cv_local(f"{DRIVE_DATA}/cv-corpus-17.0-fr-CA.tar.gz",
#                         n_test=N_TEST, n_train=N_TRAIN_FT, n_val=N_VAL_FT)
print("(Cellule optionnelle — non exécutée)")

In [ ]:
s = cv_frca["test"][0]
print("Transcription :", s["sentence"])
print(f"Durée : {len(s['audio']['array'])/SAMPLE_RATE:.2f}s")
display(Audio(s["audio"]["array"], rate=SAMPLE_RATE))

---
## 3. Analyse exploratoire <a id='section3'></a>

In [ ]:
def get_stats(ex):
    return {"duration": len(ex["audio"]["array"])/SAMPLE_RATE,
            "n_chars":  len(ex["sentence"])}

df_fr   = cv_fr["test"].map(get_stats).to_pandas()[["duration","n_chars","sentence"]]
df_frca = cv_frca["test"].map(get_stats).to_pandas()[["duration","n_chars","sentence"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col, title, xlabel in [("duration","Durées audio","Durée (s)"),
                            ("n_chars", "Longueurs",   "Nb. caractères")]:
    ax = axes[0] if col=="duration" else axes[1]
    ax.hist(df_fr[col],   bins=30, alpha=0.6, label="fr",    color='steelblue')
    ax.hist(df_frca[col], bins=30, alpha=0.6, label="fr-CA", color='tomato')
    ax.set(xlabel=xlabel, ylabel="Fréquence", title=f"Distribution des {title}")
    ax.legend()
plt.tight_layout()
save_fig("fig_eda.png"); plt.show()
print("\n── fr ──");    print(df_fr[["duration","n_chars"]].describe().round(2))
print("\n── fr-CA ──"); print(df_frca[["duration","n_chars"]].describe().round(2))

---
## 4. Évaluation baseline — Whisper <a id='section4'></a>
> ⏭️ **Sautez si §0 affiche `✅ baseline`.**

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[\.,;:!?'\"\-\(\)]", "", text)
    return re.sub(r"\s+", " ", text)

def evaluate_whisper(model_name, dataset, language="fr", batch_size=8):
    print(f"\n── {Path(model_name).name} | lang={language} | {len(dataset)} ex. ──")
    proc  = WhisperProcessor.from_pretrained(model_name, language=language, task="transcribe")
    # Charger en float32 explicitement pour éviter le mismatch fp16/fp32
    # (large-v3 a ses biais en fp16 par défaut, ce qui cause un RuntimeError)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_name, torch_dtype=torch.float32
    ).eval().to(DEVICE)
    model.config.forced_decoder_ids = None
    model.generation_config.language = language
    model.generation_config.task     = "transcribe"
    # Supprimer max_length du generation_config pour éviter le warning
    model.generation_config.max_length = None
    refs, hyps = [], []
    for i in range(0, len(dataset), batch_size):
        batch  = dataset[i: i+batch_size]
        arrays = [a["array"] for a in batch["audio"]]
        # Traiter chaque audio individuellement pour garantir 3000 bins (30s pad/trunc)
        input_list = [
            proc(a, sampling_rate=SAMPLE_RATE, return_tensors="pt",
                 return_attention_mask=False).input_features
            for a in arrays
        ]
        # input_features est float32 par défaut — cohérent avec torch_dtype=float32
        input_features = torch.cat(input_list, dim=0).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(
                input_features,
                max_new_tokens=225,
                language=language,
                task="transcribe",
            )
        hyps.extend([normalize_text(p) for p in proc.batch_decode(ids, skip_special_tokens=True)])
        refs.extend([normalize_text(r) for r in batch["sentence"]])
        if (i//batch_size) % 10 == 0:
            print(f"  {min(i+batch_size,len(dataset))}/{len(dataset)} …")
    w, c = wer(refs, hyps), cer(refs, hyps)
    print(f"  ➜ WER={w*100:.2f}%  CER={c*100:.2f}%")
    del model; torch.cuda.empty_cache()
    return {"wer": w, "cer": c, "references": refs, "hypotheses": hyps}

print("✅ Fonctions baseline définies")

In [ ]:
if baseline_results is not None:
    print("⏭️  Baseline déjà disponible depuis WandB — section sautée.")
else:
    MODELS = {
        "small"   : ("openai/whisper-small",    16),
        "medium"  : ("openai/whisper-medium",    8),
        "large-v3": ("openai/whisper-large-v3",  4),
    }
    baseline_results = {}
    _refs_hyps = {}  # conservé en mémoire pour §5, non persisté dans l'artifact

    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name="baseline-evaluation", reinit="finish_previous",
                     config={"n_test": N_TEST, "models": list(MODELS.keys())})

    for variant, (model_id, bs) in MODELS.items():
        for lang_key, ds in {"fr": cv_fr["test"], "fr-CA": cv_frca["test"]}.items():
            key = f"{variant}/{lang_key}"
            res = evaluate_whisper(model_id, ds, batch_size=bs)
            baseline_results[key] = {"wer": res["wer"], "cer": res["cer"]}
            _refs_hyps[key]       = (res["references"], res["hypotheses"])
            run.log({f"baseline/{key}/wer": res["wer"],
                     f"baseline/{key}/cer": res["cer"]})

    bl_json = f"{LOCAL_CKPT_DIR}/baseline_results.json"
    save_json(baseline_results, bl_json)
    save_artifact("baseline", "evaluation",
                  {"baseline_results.json": bl_json},
                  metadata={"n_test": N_TEST}, run=run)
    wandb.finish()
    print("\n✅ Baseline sauvegardée sur WandB")

In [ ]:
rows = [{"Variante": k.split("/")[0], "Dataset": k.split("/")[1],
         "WER (%)": round(v["wer"]*100,2), "CER (%)": round(v["cer"]*100,2)}
        for k, v in baseline_results.items()]
df_baseline = pd.DataFrame(rows)
print(df_baseline.to_string(index=False))

pivot = df_baseline.pivot(index="Variante", columns="Dataset", values="WER (%)")
ax = pivot.plot(kind="bar", figsize=(9,5), color=["steelblue","tomato"],
                edgecolor="white", width=0.55)
ax.set_title("WER baseline — Whisper (fr vs fr-CA)", fontweight="bold")
ax.set_xlabel("Variante"); ax.set_ylabel("WER (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for c in ax.containers: ax.bar_label(c, fmt='%.1f%%', padding=3, fontsize=9)
plt.tight_layout(); save_fig("fig_baseline_wer.png"); plt.show()

---
## 5. Analyse qualitative des erreurs <a id='section5'></a>

In [ ]:
from jiwer import process_words
from collections import Counter

# Si reprise depuis WandB, les refs/hyps ne sont pas dans l'artifact → recalcul léger
try:
    refs_frca, hyps_frca = _refs_hyps["small/fr-CA"]
except (NameError, KeyError):
    print("Recalcul Whisper small sur fr-CA pour l'analyse qualitative …")
    res = evaluate_whisper("openai/whisper-small", cv_frca["test"], batch_size=16)
    refs_frca, hyps_frca = res["references"], res["hypotheses"]

# Top-15 pires exemples
worst = sorted([(wer([r],[h]),r,h) for r,h in zip(refs_frca,hyps_frca)], reverse=True)
print(f"{'WER':>6}  {'Référence':<35}  Hypothèse\n" + "─"*80)
for ws, ref, hyp in worst[:15]:
    print(f"{ws:6.2f}  {(ref[:33]+'…') if len(ref)>35 else ref:<35}  {hyp[:45]}")

# Types d'erreurs
S=D=I=H=0
for ref, hyp in zip(refs_frca, hyps_frca):
    o=process_words(ref,hyp); S+=o.substitutions; D+=o.deletions; I+=o.insertions; H+=o.hits
total=S+D+I
fig, ax = plt.subplots(figsize=(5,5))
ax.pie([S,D,I], labels=["Substitutions","Suppressions","Insertions"],
       autopct='%1.1f%%', colors=["tomato","steelblue","mediumseagreen"], startangle=90)
ax.set_title("Types d'erreurs — Whisper small / fr-CA")
save_fig("fig_error_types.png"); plt.show()
print(f"Sub: {S} ({100*S/total:.1f}%) | Del: {D} ({100*D/total:.1f}%) | Ins: {I} ({100*I/total:.1f}%)")

---
## 6. Fine-tuning — Whisper small sur fr-CA <a id='section6'></a>
> ⏭️ **Sautez si §0 affiche `✅ finetuning`.**

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_FT)
tokenizer  = WhisperTokenizer.from_pretrained(MODEL_FT, language="French", task="transcribe")
processor  = WhisperProcessor.from_pretrained(MODEL_FT, language="French", task="transcribe")
metric_wer = evaluate.load("wer")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features):
        inp  = [{"input_features": f["input_features"]} for f in features]
        bat  = self.processor.feature_extractor.pad(inp, return_tensors="pt")
        lbl  = self.processor.tokenizer.pad(
                   [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        l    = lbl["input_ids"].masked_fill(lbl.attention_mask.ne(1), -100)
        if (l[:, 0] == self.decoder_start_token_id).all().cpu().item():
            l = l[:, 1:]
        bat["labels"] = l
        return bat

def compute_metrics(pred):
    pid = pred.predictions; lid = pred.label_ids
    lid[lid == -100] = tokenizer.pad_token_id
    ps = [normalize_text(p) for p in tokenizer.batch_decode(pid, skip_special_tokens=True)]
    ls = [normalize_text(l) for l in tokenizer.batch_decode(lid, skip_special_tokens=True)]
    return {"wer": 100 * metric_wer.compute(predictions=ps, references=ls)}

print("✅ Composants fine-tuning prêts")

In [ ]:
if ft_result is not None:
    print(f"⏭️  Fine-tuning déjà disponible depuis WandB.")
    print(f"   WER test = {ft_result['wer']*100:.2f}%  |  Modèle : {FT_MODEL_PATH}")
else:
    FT_LR, FT_BS, FT_EPOCHS = 1e-5, 16, 5
    FT_MODEL_PATH = f"{LOCAL_CKPT_DIR}/whisper-small-frca"

    print("Préparation des données …")
    train_ds = cv_frca["train"].map(prepare_dataset,
        remove_columns=cv_frca["train"].column_names, num_proc=2)
    val_ds   = cv_frca["validation"].map(prepare_dataset,
        remove_columns=cv_frca["validation"].column_names, num_proc=2)

    model_ft = WhisperForConditionalGeneration.from_pretrained(MODEL_FT)
    model_ft.config.forced_decoder_ids = None
    model_ft.config.suppress_tokens    = []
    collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model_ft.config.decoder_start_token_id)

    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f"fine-tuning-lr{FT_LR}", reinit="finish_previous",
                     config={"lr": FT_LR, "epochs": FT_EPOCHS, "batch_size": FT_BS,
                             "model": MODEL_FT, "dataset": "cv13-fr-CA"})

    trainer = Seq2SeqTrainer(
        model=model_ft, args=Seq2SeqTrainingArguments(
            output_dir=FT_MODEL_PATH,
            per_device_train_batch_size=FT_BS, per_device_eval_batch_size=8,
            learning_rate=FT_LR, warmup_steps=500,
            num_train_epochs=FT_EPOCHS,
            gradient_checkpointing=True, fp16=True,
            evaluation_strategy="epoch", save_strategy="epoch",
            load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
            predict_with_generate=True, generation_max_length=225,
            logging_steps=25, report_to=["wandb"],
            run_name=f"whisper-small-frca-lr{FT_LR}",
            push_to_hub=False, label_names=["labels"],
        ),
        train_dataset=train_ds, eval_dataset=val_ds,
        tokenizer=processor.feature_extractor,
        data_collator=collator, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    trainer.save_model()

    # Courbes d'apprentissage
    lt = [l for l in trainer.state.log_history if "loss" in l and "eval_loss" not in l]
    le = [l for l in trainer.state.log_history if "eval_wer" in l]
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    if lt: axes[0].plot([l["step"] for l in lt], [l["loss"] for l in lt], color="steelblue")
    axes[0].set(title="Train Loss", xlabel="Step", ylabel="Loss")
    if le: axes[1].plot([l["epoch"] for l in le], [l["eval_wer"] for l in le],
                        marker='o', color="tomato")
    axes[1].set(title="WER validation", xlabel="Époque", ylabel="WER (%)")
    plt.tight_layout()
    fig_curves = save_fig("fig_training_curves.png"); plt.show()

    # Évaluation finale test
    res_ft    = evaluate_whisper(FT_MODEL_PATH, cv_frca["test"], batch_size=8)
    ft_result = {"wer": res_ft["wer"], "cer": res_ft["cer"]}
    run.log({"test/wer": ft_result["wer"], "test/cer": ft_result["cer"]})

    # ── Sauvegarde artifact (JSON + modèle + figure) ──────────────────────────
    ft_json = f"{LOCAL_CKPT_DIR}/ft_result.json"
    save_json(ft_result, ft_json)
    save_artifact("finetuning", "model",
                  {"ft_result.json":          ft_json,
                   "model":                   FT_MODEL_PATH,
                   "fig_training_curves.png": fig_curves},
                  metadata={"wer": ft_result["wer"], "lr": FT_LR, "epochs": FT_EPOCHS},
                  run=run)
    wandb.finish()
    del model_ft; torch.cuda.empty_cache()
    print(f"\n✅ Fine-tuning terminé | WER test = {ft_result['wer']*100:.2f}%")

In [ ]:
bl = baseline_results["small/fr-CA"]
print(f"{'':26} {'WER (%)':>8}  {'CER (%)':>8}\n" + "─"*46)
print(f"{'Whisper small baseline':26} {bl['wer']*100:8.2f}  {bl['cer']*100:8.2f}")
print(f"{'Whisper small fine-tuné':26} {ft_result['wer']*100:8.2f}  {ft_result['cer']*100:8.2f}")
delta = (ft_result['wer'] - bl['wer']) * 100
print(f"\nΔ WER : {delta:+.2f} pp  |  Réduction relative : {delta/bl['wer']/100*100:.1f}%")

---
## 7. Ablations & hyperparamètres <a id='section7'></a>
> ⏭️ Chaque sous-section vérifie **son propre artifact** indépendamment.

In [ ]:
# Datasets & collator — recréés si on arrive ici après une reprise
try:
    _ = train_ds
except NameError:
    print("Préparation des datasets pour ablations …")
    train_ds = cv_frca["train"].map(prepare_dataset,
                   remove_columns=cv_frca["train"].column_names, num_proc=2)
    val_ds   = cv_frca["validation"].map(prepare_dataset,
                   remove_columns=cv_frca["validation"].column_names, num_proc=2)
    _tmp = WhisperForConditionalGeneration.from_pretrained(MODEL_FT)
    collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor, decoder_start_token_id=_tmp.config.decoder_start_token_id)
    del _tmp
    print("✅ Datasets prêts")

def run_ablation(label, lr=1e-5, freeze_enc=False, train_data=None, n_epochs=2):
    """Lance un fine-tuning d'ablation (2 époques) et retourne le WER de validation."""
    m = WhisperForConditionalGeneration.from_pretrained(MODEL_FT)
    m.config.forced_decoder_ids = None; m.config.suppress_tokens = []
    if freeze_enc:
        for p in m.model.encoder.parameters(): p.requires_grad = False
        print(f"  Params entraînables: {sum(p.numel() for p in m.parameters() if p.requires_grad):,}")
    t = Seq2SeqTrainer(
        model=m, args=Seq2SeqTrainingArguments(
            output_dir=f"/content/abl_{label}",
            per_device_train_batch_size=16, per_device_eval_batch_size=8,
            learning_rate=lr, warmup_steps=200, num_train_epochs=n_epochs,
            fp16=True, evaluation_strategy="epoch", save_strategy="no",
            predict_with_generate=True, generation_max_length=225,
            logging_steps=50, report_to=["wandb"],
            run_name=f"ablation-{label}", label_names=["labels"]),
        train_dataset=train_data or train_ds, eval_dataset=val_ds,
        tokenizer=processor.feature_extractor,
        data_collator=collator, compute_metrics=compute_metrics,
    )
    t.train()
    res = t.evaluate()
    del m, t; torch.cuda.empty_cache()
    return res["eval_wer"]

print("✅ Fonction d'ablation prête")

In [ ]:
# ── Ablation 1 : Taux d'apprentissage ─────────────────────────────────────────
if ablation_lr_results is not None:
    print("⏭️  Ablation LR déjà disponible depuis WandB :")
    for lr, w in ablation_lr_results.items(): print(f"  LR={lr} → WER={w:.2f}%")
else:
    LEARNING_RATES   = [1e-5, 3e-5, 5e-5]
    ablation_lr_results = {}
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name="ablation-lr", reinit="finish_previous")
    for lr in LEARNING_RATES:
        w = run_ablation(f"lr{lr}", lr=lr)
        ablation_lr_results[lr] = w
        run.log({f"ablation/lr/{lr}/wer": w})
        print(f"  LR={lr} → WER={w:.2f}%")
    abl_json = f"{LOCAL_CKPT_DIR}/ablation_lr.json"
    save_json({str(k): v for k,v in ablation_lr_results.items()}, abl_json)
    save_artifact("ablation-lr", "ablation",
                  {"ablation_lr.json": abl_json},
                  metadata={"learning_rates": list(ablation_lr_results.keys())}, run=run)
    wandb.finish()

fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar([str(lr) for lr in ablation_lr_results],
              list(ablation_lr_results.values()),
              color="steelblue", edgecolor="white", width=0.5)
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set(xlabel="Taux d'apprentissage", ylabel="WER val (%)",
       title="Ablation : taux d'apprentissage")
save_fig("fig_ablation_lr.png"); plt.show()

In [ ]:
# ── Ablation 2 : Gel de l'encodeur ────────────────────────────────────────────
if ablation_freeze_results is not None:
    print("⏭️  Ablation freeze déjà disponible depuis WandB :")
    for k, v in ablation_freeze_results.items(): print(f"  {k} → WER={v:.2f}%")
else:
    ablation_freeze_results = {}
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name="ablation-freeze", reinit="finish_previous")
    for freeze, label in [(False,"encodeur_libre"),(True,"encodeur_gelé")]:
        w = run_ablation(label, lr=1e-5, freeze_enc=freeze)
        ablation_freeze_results[label] = w
        run.log({f"ablation/freeze/{label}/wer": w})
        print(f"  {label} → WER={w:.2f}%")
    frz_json = f"{LOCAL_CKPT_DIR}/ablation_freeze.json"
    save_json(ablation_freeze_results, frz_json)
    save_artifact("ablation-freeze", "ablation",
                  {"ablation_freeze.json": frz_json}, run=run)
    wandb.finish()

In [ ]:
# ── Ablation 3 : Augmentation audio ───────────────────────────────────────────
if ablation_aug_wer is not None:
    print(f"⏭️  Ablation augmentation déjà disponible : WER={ablation_aug_wer:.2f}%")
else:
    from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
    augment = Compose([
        AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
        TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
        PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
    ])
    def prepare_aug(batch):
        audio = batch["audio"]
        arr   = augment(samples=audio["array"].astype(np.float32),
                        sample_rate=audio["sampling_rate"])
        batch["input_features"] = feature_extractor(
            arr, sampling_rate=audio["sampling_rate"]).input_features[0]
        batch["labels"] = tokenizer(batch["sentence"]).input_ids
        return batch
    train_aug = cv_frca["train"].map(prepare_aug,
                    remove_columns=cv_frca["train"].column_names)

    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name="ablation-augmentation", reinit="finish_previous")
    ablation_aug_wer = run_ablation("augmented", lr=1e-5, train_data=train_aug)
    run.log({"ablation/augment/wer": ablation_aug_wer})

    aug_json = f"{LOCAL_CKPT_DIR}/ablation_augment.json"
    save_json({"wer": ablation_aug_wer}, aug_json)
    save_artifact("ablation-augment", "ablation",
                  {"ablation_augment.json": aug_json},
                  metadata={"wer": ablation_aug_wer}, run=run)
    wandb.finish()
    print(f"✅ WER augmentation = {ablation_aug_wer:.2f}%")

In [ ]:
# ── Tableau récapitulatif des ablations ───────────────────────────────────────
bl_wer = baseline_results["small/fr-CA"]["wer"] * 100

abl_rows = [
    {"Expérience": "Baseline (sans FT)",         "WER val (%)": round(bl_wer, 2),      "Δ pp": "—"},
    {"Expérience": "FT lr=1e-5",  "WER val (%)": round(ablation_lr_results.get(1e-5, float('nan')),2),
     "Δ pp": f"{ablation_lr_results.get(1e-5,0)-bl_wer:+.2f}"},
    {"Expérience": "FT lr=3e-5",  "WER val (%)": round(ablation_lr_results.get(3e-5, float('nan')),2),
     "Δ pp": f"{ablation_lr_results.get(3e-5,0)-bl_wer:+.2f}"},
    {"Expérience": "FT lr=5e-5",  "WER val (%)": round(ablation_lr_results.get(5e-5, float('nan')),2),
     "Δ pp": f"{ablation_lr_results.get(5e-5,0)-bl_wer:+.2f}"},
    {"Expérience": "FT encodeur libre", "WER val (%)": round(ablation_freeze_results.get('encodeur_libre',float('nan')),2),
     "Δ pp": f"{ablation_freeze_results.get('encodeur_libre',0)-bl_wer:+.2f}"},
    {"Expérience": "FT encodeur gelé",  "WER val (%)": round(ablation_freeze_results.get('encodeur_gelé',float('nan')),2),
     "Δ pp": f"{ablation_freeze_results.get('encodeur_gelé',0)-bl_wer:+.2f}"},
    {"Expérience": "FT + augmentation audio", "WER val (%)": round(ablation_aug_wer,2),
     "Δ pp": f"{ablation_aug_wer-bl_wer:+.2f}"},
]
abl_df = pd.DataFrame(abl_rows)
print("\n══════════ TABLEAU D'ABLATIONS ══════════")
print(abl_df.to_string(index=False))
abl_df.to_csv(f"{LOCAL_CKPT_DIR}/ablation_results.csv", index=False)

---
## 8. Comparaison avec wav2vec 2.0 <a id='section8'></a>
> ⏭️ **Sautez si §0 affiche `✅ wav2vec`.**

In [ ]:
def evaluate_wav2vec(model_name, dataset, batch_size=8):
    print(f"\n── wav2vec 2.0 | {len(dataset)} ex. ──")
    proc  = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2ForCTC.from_pretrained(
        model_name, torch_dtype=torch.float32
    ).eval().to(DEVICE)
    refs, hyps = [], []
    for i in range(0, len(dataset), batch_size):
        batch  = dataset[i: i+batch_size]
        inputs = proc([a["array"] for a in batch["audio"]],
                      sampling_rate=SAMPLE_RATE, return_tensors="pt",
                      padding=True).to(DEVICE)
        # S'assurer que les inputs sont en float32
        inputs = {k: v.float() if v.dtype == torch.float16 else v
                  for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        hyps.extend([normalize_text(p) for p in proc.batch_decode(torch.argmax(logits,-1))])
        refs.extend([normalize_text(r) for r in batch["sentence"]])
        if (i//batch_size) % 10 == 0:
            print(f"  {min(i+batch_size,len(dataset))}/{len(dataset)} …")
    w, c = wer(refs, hyps), cer(refs, hyps)
    print(f"  WER={w*100:.2f}%  CER={c*100:.2f}%")
    del model; torch.cuda.empty_cache()
    return {"wer": w, "cer": c}


---
## 9. Synthèse des résultats <a id='section9'></a>

In [ ]:
rows = []
for variant in ["small","medium","large-v3"]:
    for lang in ["fr","fr-CA"]:
        res = baseline_results.get(f"{variant}/{lang}")
        if res:
            rows.append({"Modèle": f"Whisper {variant}", "Fine-tuning": "Non",
                         "Dataset": lang, "WER (%)": round(res["wer"]*100,2),
                         "CER (%)": round(res["cer"]*100,2)})
if ft_result:
    rows.append({"Modèle": "Whisper small", "Fine-tuning": "Oui (fr-CA)",
                 "Dataset": "fr-CA",
                 "WER (%)": round(ft_result["wer"]*100,2),
                 "CER (%)": round(ft_result["cer"]*100,2)})
for lang, res in [("fr",w2v_fr),("fr-CA",w2v_frca)]:
    if res:
        rows.append({"Modèle": "wav2vec 2.0 XLSR-53", "Fine-tuning": "Non",
                     "Dataset": lang,
                     "WER (%)": round(res["wer"]*100,2),
                     "CER (%)": round(res["cer"]*100,2)})

df_synth = pd.DataFrame(rows)
print("═"*50 + "\n     SYNTHÈSE DES RÉSULTATS\n" + "═"*50)
print(df_synth.to_string(index=False))
df_synth.to_csv(f"{LOCAL_CKPT_DIR}/synthesis_results.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))
for ax, metric in zip(axes, ["WER (%)","CER (%)"]):
    pivot = df_synth.pivot_table(index="Modèle", columns="Dataset", values=metric, aggfunc="mean")
    pivot.plot(kind="barh", ax=ax, color=["steelblue","tomato"], edgecolor="white")
    ax.set_title(f"{metric[:3]} — Comparaison tous modèles", fontweight="bold")
    ax.set_xlabel(metric); ax.set_ylabel("")
    for c in ax.containers: ax.bar_label(c, fmt='%.1f', padding=3, fontsize=8)
plt.suptitle("INF8225 — ASR : Whisper vs wav2vec 2.0 (fr / fr-CA)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig("fig_synthesis.png"); plt.show()

In [ ]:
# ── Upload final de toutes les figures sur WandB ──────────────────────────────
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name="figures-final", reinit="finish_previous")
figs = glob.glob(f"{FIG_DIR}/fig_*.png")
art  = wandb.Artifact(_art_name("figures"), type="figures")
for f in figs:
    art.add_file(f)
    run.log({Path(f).stem: wandb.Image(f)})
run.log_artifact(art)
wandb.finish()

print("\n✅ Expériences terminées.")
print(f"   {len(figs)} figures uploadées sur WandB")
print(f"   CSV de synthèse : {LOCAL_CKPT_DIR}/synthesis_results.csv")
print("   Données prêtes pour le rapport IJCAI et les diapositives.")